# Aula 01 — Fundamentos de NLP · acompanhamento

Notebook para rodar célula a célula enquanto você assiste à vídeo-aula. Os blocos seguem a ordem dos slides: **30 → 49 → 52–56**.

**Origem:** `IAST-ProcessamentodeLinguagemNatural-Fase04_aula01-v1.pptx`

**Kernel:** selecione o `.venv` do projeto (`POSTECH_AI_SCIENTIST/.venv`). Sem ele, `nltk` e `spacy` não são encontrados.

**Instalação (uma vez só, no terminal da raiz do projeto):**

```powershell
py -3.14 -m venv .venv
.venv\Scripts\python -m pip install nltk spacy scikit-learn pandas matplotlib ipykernel
.venv\Scripts\python -m spacy download pt_core_news_lg
```

> **Uma correção ao código do slide.** O slide 52 baixa apenas `punkt`. A partir do NLTK 3.8.2 o `word_tokenize` exige `punkt_tab` — sem ele o slide 53 quebra com `LookupError`. A linha original do professor está preservada, comentada, ao lado da corrigida.

---
## Bloco 1 · Slide 30 — Polissemia em código

**O conceito:** "banco" tem três sentidos distintos em português — financeiro, móvel, tecnológico. O BoW conta ocorrências de strings e não tem nenhuma noção de sentido, então as três frases produzem o mesmo `1` na coluna `banco`.

**Por que isso importa:** toda a família BoW/TF-IDF (aulas 2 e 3) carrega essa limitação. É exatamente o problema que os embeddings contextuais resolvem — a razão de BERT e afins existirem. Guarde esta célula: é o contraexemplo que justifica metade da disciplina.

In [1]:
# A palavra "banco" tem múltiplos sentidos em português
from sklearn.feature_extraction.text import CountVectorizer

corpus = [
    "O banco aprovou meu empréstimo ontem",       # financeiro
    "Sentei no banco da praça para descansar",     # móvel
    "O banco de dados PostgreSQL caiu novamente",  # tech
]
vec = CountVectorizer()
X = vec.fit_transform(corpus)
print('Vocabulário:', vec.get_feature_names_out())
# Para o BoW, 'banco' aparece igual nos 3 documentos.
# Não há nada que distinga sentido financeiro, móvel ou tech!
print('Matriz documento-termo:')
print(X.toarray())
# A coluna de 'banco' tem 1 em todas as linhas → polissemia perdida

Vocabulário: ['aprovou' 'banco' 'caiu' 'da' 'dados' 'de' 'descansar' 'empréstimo' 'meu'
 'no' 'novamente' 'ontem' 'para' 'postgresql' 'praça' 'sentei']
Matriz documento-termo:
[[1 1 0 0 0 0 0 1 1 0 0 1 0 0 0 0]
 [0 1 0 1 0 0 1 0 0 1 0 0 1 0 1 1]
 [0 1 1 0 1 1 0 0 0 0 1 0 0 1 0 0]]


### Suas anotações

_(o que o professor falou aqui)_

---
## Bloco 2 · Slide 49 — Lei de Zipf

> Este slide não tem código executável — é um bloco de conceito. O conteúdo original está reproduzido abaixo; você vê a lei acontecendo de fato no slide 53.

A frequência de um token é inversamente proporcional à sua posição no ranking. Num corpus de português:

| token | ~% de todos os tokens |
|---|---|
| `de` | ~5% |
| `a` | ~3% |
| `o` | ~3% |
| `que` | ~2% |

- Palavras como *arqueólogo* aparecem 1–2 vezes.
- **50% do vocabulário aparece uma única vez** (*hapax legomena*).

**Implicação prática:**

- Top 100 palavras ≈ 50% de todos os tokens
- Top 1.000 palavras ≈ 80%
- Resto do vocabulário (milhares de palavras) ≈ 20%

**Por que isso importa:** é a justificativa estatística para remover stop words e para os filtros `min_df` / `max_df` da aula 3. A cauda longa de termos raríssimos infla o vocabulário sem agregar sinal.

### Suas anotações

_(o que o professor falou aqui)_

---
## Bloco 3 · Slides 52–56 — o pipeline

As cinco células a seguir **compartilham estado** e precisam rodar em ordem: `corpus`, `nlp` e `Counter` nascem no slide 52 e são usados nos slides 53 e 56.

### Slide 52 — Setup do ambiente

**O conceito:** as duas bibliotecas resolvem problemas diferentes. NLTK é a caixa de ferramentas acadêmica — tokenizadores, stemmers, corpora avulsos. spaCy é o pipeline de produção: carrega um modelo treinado e entrega tokenização, POS, lema e NER numa passada só.

Repare no corpus: são cinco *reviews* de produto, e a última — *"Esperei 3 horas no telefone. Adorei o atendimento."* — é irônica de propósito. Guarde isso, porque nenhuma técnica desta aula vai capturar a ironia.

⚠️ Esta célula **redefine `corpus`** — o slide 30 usou o mesmo nome para outro conjunto. Se voltar a rodar o Bloco 1 depois daqui, rode esta célula de novo antes de seguir, ou o slide 56 vetoriza o corpus errado.

In [1]:
# Instalar bibliotecas
# pip install nltk spacy scikit-learn
# python -m spacy download pt_core_news_lg
import nltk
import spacy
import pandas as pd
from collections import Counter

# Recursos do NLTK em português
# nltk.download('punkt')     # linha original do slide — ver nota no topo do notebook
nltk.download('punkt_tab')   # exigido pelo word_tokenize a partir do NLTK 3.8.2
nltk.download('stopwords')
nltk.download('rslp')          # stemmer português

# Carregar modelo do spaCy em português
nlp = spacy.load('pt_core_news_lg')

# Corpus exemplo: reviews fictícias de produtos
corpus = [
    "O produto é excelente, recomendo muito! Chegou rápido.",
    "Péssimo, o aparelho veio com defeito. Não comprem.",
    "Ótima qualidade pelo preço. Entrega rápida e bem embalado.",
    "Adorei! Funciona perfeitamente, superou minhas expectativas.",
    "Esperei 3 horas no telefone. Adorei o atendimento.",  # ironia
]

[nltk_data] Downloading package punkt_tab to
[nltk_data]     ~\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     ~\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package rslp to
[nltk_data]     ~\AppData\Roaming\nltk_data...
[nltk_data]   Package rslp is already up-to-date!


### Suas anotações

_(o que o professor falou aqui)_

### Slide 53 — Passo 1: tokenização e contagem (NLTK)

**O conceito:** a Lei de Zipf do bloco anterior, agora visível. Contando tokens crus, o topo do ranking é ocupado por pontuação e stop words (`o`, `.`, `e`, `,`, `com`, `a`) — nenhuma delas diz nada sobre o produto ser bom ou ruim.

É este resultado que motiva o `stop_words=` do slide 56.

In [2]:
from nltk.tokenize import word_tokenize

# Tokenizar todos os documentos
todos_tokens = []
for doc in corpus:
    tokens = word_tokenize(doc.lower(), language='portuguese')
    todos_tokens.extend(tokens)
print(f'Total de tokens: {len(todos_tokens)}')
print(f'Vocabulário (únicos): {len(set(todos_tokens))}')

# Top 10 palavras mais frequentes (Lei de Zipf)
contador = Counter(todos_tokens)
print('\nTop 10 mais frequentes:')
for palavra, freq in contador.most_common(10):
    print(f'  {palavra:15s} → {freq}')
# Saída esperada:
# o, ., e, , o, com, a (stop words dominam)

Total de tokens: 52
Vocabulário (únicos): 39

Top 10 mais frequentes:
  .               → 8
  o               → 3
  ,               → 3
  !               → 2
  adorei          → 2
  produto         → 1
  é               → 1
  excelente       → 1
  recomendo       → 1
  muito           → 1


### Suas anotações

_(o que o professor falou aqui)_

### Slide 54 — Passo 2: análise linguística (spaCy)

**O conceito:** aqui o texto deixa de ser uma lista de strings. Cada token ganha lema (`comprou` → `comprar`), classe gramatical (POS) e função sintática (DEP).

O lema é o que permite tratar `comprou`, `comprei` e `comprar` como o mesmo termo na vetorização — reduz o vocabulário sem perder sinal, ao contrário do stemming, que corta a palavra no braço. A comparação stemming × lemmatization é a aula 2.

Repare em `na` → lema `em o`: o spaCy desfaz a contração.

In [3]:
# spaCy oferece tokenização robusta + POS tagging + lemmatization + NER
doc = nlp("O João comprou um livro na Amazon ontem.")
print(f"{'TOKEN':12s} {'LEMA':12s} {'POS':10s} {'DEP':10s}")
print('-' * 50)
for token in doc:
    print(f"{token.text:12s} {token.lemma_:12s} "
          f"{token.pos_:10s} {token.dep_:10s}")
# Saída esperada:
# TOKEN        LEMA         POS        DEP
# O            o            DET        det
# João         João         PROPN      nsubj
# comprou      comprar      VERB       ROOT
# um           um           DET        det
# livro        livro        NOUN       obj
# na           em o         ADP        case
# Amazon       Amazon       PROPN      obl
# ontem        ontem        ADV        advmod
# .            .            PUNCT      punct

TOKEN        LEMA         POS        DEP       
--------------------------------------------------
O            o            DET        det       
João         João         PROPN      nsubj     
comprou      comprar      VERB       ROOT      
um           um           DET        det       
livro        livro        NOUN       obj       
na           em o         ADP        case      
Amazon       Amazon       PROPN      obl       
ontem        ontem        ADV        advmod    
.            .            PUNCT      punct     


### Suas anotações

_(o que o professor falou aqui)_

### Slide 55 — NER: reconhecimento de entidades nomeadas

**O conceito:** o modelo classifica trechos do texto em pessoa (`PER`), local (`LOC`), organização (`ORG`) e diversos (`MISC`) — sem nenhuma lista fixa, a partir do que foi treinado.

**Por que isso importa para o Tech Challenge:** é assim que se extrai metadado de texto livre. Numa base de reviews, NER é o caminho para separar menções a marcas, transportadoras e locais do resto do comentário.

As saídas variam um pouco entre versões do modelo — se a sua não bater 100% com a do slide, não é erro seu.

In [4]:
# Identificação de entidades nomeadas em português
texto = ("Leonardo da Vinci nasceu em Vinci, Itália, em 1452. "
        "Foi pintor, cientista e trabalhou para Lorenzo de Médici. "
        "Morreu em Amboise, na França.")
doc = nlp(texto)
print('Entidades encontradas:')
for ent in doc.ents:
    print(f'  {ent.text:25s} → {ent.label_}')
# Saída esperada:
# Leonardo da Vinci         → PER  (pessoa)
# Vinci                     → LOC  (local)
# Itália                    → LOC
# 1452                      → MISC
# Lorenzo de Médici         → PER
# Amboise                   → LOC
# França                    → LOC

Entidades encontradas:
  Leonardo da Vinci         → PER
  Vinci                     → LOC
  Itália                    → LOC
  Lorenzo de Médici         → PER
  Amboise                   → LOC
  França                    → LOC


### Suas anotações

_(o que o professor falou aqui)_

### Slide 56 — Passo 3: primeira vetorização

**O conceito:** aqui o texto vira número e o resto da disciplina fica possível. A matriz documento-termo tem uma linha por documento e uma coluna por termo do vocabulário.

Repare no `shape`: 5 documentos e N colunas, com a maioria das células em zero — se cada review tem ~8 palavras úteis e o vocabulário tem dezenas de termos, quase tudo é zero. Essa esparsidade é o assunto dos slides 18–19 da aula 3, e o motivo de o scikit-learn devolver uma matriz esparsa do SciPy em vez de um array denso.

O `stop_words=stopwords_pt` é a resposta direta ao que você viu no slide 53.

In [5]:
from sklearn.feature_extraction.text import CountVectorizer
from nltk.corpus import stopwords

stopwords_pt = stopwords.words('portuguese')

# Bag of Words com remoção de stop words
vec = CountVectorizer(stop_words=stopwords_pt)
X = vec.fit_transform(corpus)
print(f'Shape da matriz: {X.shape}')  # (5 docs, N features)
print(f'Vocabulário ({len(vec.vocabulary_)} termos):')
print(vec.get_feature_names_out()[:15])

# Visualizar matriz documento-termo (esparsa!)
import pandas as pd
df_dt = pd.DataFrame(
    X.toarray(),
    columns=vec.get_feature_names_out()
)
print('\nMatriz documento-termo (primeiras colunas):')
print(df_dt.iloc[:, :8])

Shape da matriz: (5, 26)
Vocabulário (26 termos):
['adorei' 'aparelho' 'atendimento' 'bem' 'chegou' 'comprem' 'defeito'
 'embalado' 'entrega' 'esperei' 'excelente' 'expectativas' 'funciona'
 'horas' 'perfeitamente']

Matriz documento-termo (primeiras colunas):
   adorei  aparelho  atendimento  bem  chegou  comprem  defeito  embalado
0       0         0            0    0       1        0        0         0
1       0         1            0    0       0        1        1         0
2       0         0            0    1       0        0        0         1
3       1         0            0    0       0        0        0         0
4       1         0            1    0       0        0        0         0


### Suas anotações

_(o que o professor falou aqui)_

---
## O que esta aula deixa em aberto

| Limitação vista aqui | Onde é resolvida |
|---|---|
| BoW não distingue sentidos (slide 30) | embeddings contextuais — aula 4 |
| Stop words dominam a contagem (slide 53) | remoção + `min_df`/`max_df` — aulas 2 e 3 |
| Contagem crua trata todo termo como igualmente informativo | TF-IDF — aula 3 |
| A ironia da review 5 passa batido | nenhuma técnica clássica resolve; é limite conhecido |

**Conexão com o Tech Challenge:** o corpus desta aula são cinco reviews de produto — exatamente o tipo de dado do desafio da fase (reviews do Olist). Limpeza, tokenização e vetorização são o primeiro estágio do pipeline RAG que você vai construir.